In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [34]:
import torch
import random

MAX_LENGTH = 10

subjects = [("i am", "je suis"), ("he is", "il est"), ("she is", "elle est"), ("we are", "nous sommes")]
adjectives = [("happy", "heureux"), ("sad", "triste"), ("hungry", "affame"), ("tired", "fatigue"), ("ready", "pret")]

all_data = []
for eng_subj, fra_subj in subjects:
    for eng_adj, fra_adj in adjectives:
        all_data.append((f"{eng_subj} {eng_adj}", f"{fra_subj} {fra_adj}"))

test_sentences_held_out = [
    "i am ready",     
    "he is tired",    
    "she is hungry",  
    "we are happy"    
]

train_data = []
test_data = []

for eng, fra in all_data:
    if eng in test_sentences_held_out:
        test_data.append((eng, fra))
    else:
        train_data.append((eng, fra))

print(f"Generated {len(all_data)} total sentences.")
print(f"Training on {len(train_data)} sentences. Testing on {len(test_data)} unseen sentences.")

SOS_token = 0  
EOS_token = 1  
UNK_token = 2  

def build_vocab(sentences):
    vocab = {"<SOS>": SOS_token, "<EOS>": EOS_token, "<UNK>": UNK_token}
    for sentence in sentences:
        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

# Build vocabulary
eng_sentences = [pair[0] for pair in all_data]
fra_sentences = [pair[1] for pair in all_data]

eng_vocab = build_vocab(eng_sentences)
fra_vocab = build_vocab(fra_sentences)

eng_idx2word = {i: w for w, i in eng_vocab.items()}
fra_idx2word = {i: w for w, i in fra_vocab.items()}

def tensorFromSentence(vocab, sentence):
    indexes = [vocab.get(word, UNK_token) for word in sentence.split()]
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long).view(-1, 1)

training_pairs = [(tensorFromSentence(eng_vocab, pair[0]), 
                   tensorFromSentence(fra_vocab, pair[1])) for pair in train_data]

print(f"English Vocab Size: {len(eng_vocab)}")
print(f"French Vocab Size: {len(fra_vocab)}")

Generated 20 total sentences.
Training on 16 sentences. Testing on 4 unseen sentences.
English Vocab Size: 15
French Vocab Size: 15


In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        
        # Turns word index into a dense vector (embedding)
        self.embedding = nn.Embedding(input_size, hidden_size)
        # GRU replaces standard RNN to avoid vanishing gradients
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output, hidden = self.gru(embedded, hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size
        
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = torch.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

In [ ]:
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

hidden_size = 64
encoder = Encoder(len(eng_vocab), hidden_size).to(device)
decoder = Decoder(hidden_size, len(fra_vocab)).to(device)

criterion = nn.NLLLoss()
encoder_optimizer = optim.SGD(encoder.parameters(), lr=0.01)
decoder_optimizer = optim.SGD(decoder.parameters(), lr=0.01)

epochs = 150 
print("Starting Training...")

for epoch in range(epochs):
    epoch_loss = 0
    start_time = time.time()
    total_pairs = len(training_pairs)
    
    for input_tensor, target_tensor in training_pairs:
        input_tensor = input_tensor.to(device)
        target_tensor = target_tensor.to(device)
        
        encoder_hidden = encoder.initHidden().to(device)
        
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()
        
        input_length = input_tensor.size(0)
        target_length = target_tensor.size(0)
        loss = 0
        
        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
            
        decoder_input = torch.tensor([[SOS_token]]).to(device)
        decoder_hidden = encoder_hidden
        
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]
                
        loss.backward()
        encoder_optimizer.step()
        decoder_optimizer.step()
        
        pair_loss = loss.item() / target_length
        epoch_loss += pair_loss
        
    time_taken = time.time() - start_time
    
    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {epoch_loss/total_pairs:.4f} | Time: {time_taken:.2f}s")
print("Training Complete.")

Training on device: cuda
Starting Training...
Epoch 25/150 | Avg Loss: 0.7880 | Time: 0.60s
Epoch 50/150 | Avg Loss: 0.4967 | Time: 0.57s
Epoch 75/150 | Avg Loss: 0.2716 | Time: 0.55s
Epoch 100/150 | Avg Loss: 0.0844 | Time: 0.74s
Epoch 125/150 | Avg Loss: 0.0402 | Time: 0.57s
Epoch 150/150 | Avg Loss: 0.0256 | Time: 0.72s
Training Complete.


In [ ]:
def evaluate(sentence):
    with torch.no_grad():
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        input_tensor = tensorFromSentence(eng_vocab, sentence).to(device)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden().to(device)

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)

        decoder_input = torch.tensor([[SOS_token]]).to(device)
        decoder_hidden = encoder_hidden
        decoded_words = []

        for di in range(MAX_LENGTH): 
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.data.topk(1)
            
            if topi.item() == EOS_token:
                break
            else:
                decoded_words.append(fra_idx2word.get(topi.item(), "<UNK>"))
                
            decoder_input = topi.squeeze().detach()

        return ' '.join(decoded_words)

print("\n--- Evaluation & Validation ---")
extended_evaluation_list = [
    # The 4 strictly unseen test sentences
    ("i am ready", "je suis pret", "UNSEEN"),
    ("he is tired", "il est fatigue", "UNSEEN"),
    ("she is hungry", "elle est affame", "UNSEEN"),
    ("we are happy", "nous sommes heureux", "UNSEEN"),
    
    # 6 sentences the model saw during training
    ("i am happy", "je suis heureux", "TRAIN"),
    ("he is sad", "il est triste", "TRAIN"),
    ("she is tired", "elle est fatigue", "TRAIN"),
    ("we are ready", "nous sommes pret", "TRAIN"),
    ("i am hungry", "je suis affame", "TRAIN"),
    ("he is ready", "il est pret", "TRAIN"),
]

for eng, expected_fra, split_type in extended_evaluation_list:
    prediction = evaluate(eng.lower().strip())
    print(f"[{split_type}] Input: {eng}")
    print(f"         Target:  {expected_fra}")
    print(f"         Predict: {prediction}")
    print("-" * 40)


--- Evaluation & Validation ---
[UNSEEN] Input: i am ready
         Target:  je suis pret
         Predict: je suis pret
----------------------------------------
[UNSEEN] Input: he is tired
         Target:  il est fatigue
         Predict: il est fatigue
----------------------------------------
[UNSEEN] Input: she is hungry
         Target:  elle est affame
         Predict: il est affame
----------------------------------------
[UNSEEN] Input: we are happy
         Target:  nous sommes heureux
         Predict: il est heureux
----------------------------------------
[TRAIN] Input: i am happy
         Target:  je suis heureux
         Predict: je suis heureux
----------------------------------------
[TRAIN] Input: he is sad
         Target:  il est triste
         Predict: il est triste
----------------------------------------
[TRAIN] Input: she is tired
         Target:  elle est fatigue
         Predict: elle est fatigue
----------------------------------------
[TRAIN] Input: we ar